# Sprint 53 · Task B · Step B5 — combined prompt C (Stage 1b) + held-out confirmation (Stage 2)

**Every session:** Runtime → Change runtime type → **T4 GPU**, then run cells 1 → 5.

| Cell | What it does | When |
|---|---|---|
| 1 | GPU check | every session |
| 2 | Upload zip(s) and unpack | every session |
| 3 | Install + start Ollama, download both judges, check the new prompts | every session (~3–6 min) |
| 4 | Progress of Stage 1b and Stage 2 | every session |
| 5 | Define `save_checkpoint()` | every session |
| 6 | **Part 1a** — smoke test of the 10 new prompts (20 calls) + time estimates | once |
| 7 | **Part 1b** — Stage 1b: prompt C on the development half (436 calls, ~20 min) + analysis → decides whether C joins Stage 2 | once, right after cell 6 |
| 8 | **Part 2** — Stage 2 held-out confirmation, 30-min chunks with checkpoint downloads | ⚠️ only after Claude confirms the cell 6–7 output |
| 9 | Manual checkpoint | before closing the tab |


In [1]:
# 1. GPU check - should show a Tesla T4 with ~15 GB
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


**Cell 2 — what to upload** (one file at a time; the cell keeps asking until it has everything)
* **First time:** your newest Stage 1 checkpoint (`sprint53_taskB_run_20261003_0419.zip`, or `sprint53_taskB_B4_<time>.zip` if you ran the optional analysis notebook), then `sprint53_taskB_step5_code.zip`.
* **Later sessions:** only the newest `sprint53_taskB_run_<time>.zip` checkpoint (it already contains the Step B5 code).


In [2]:
# 2. Upload and unpack
import os, zipfile, glob
from google.colab import files
os.chdir('/content')

def unpack(path):
    with zipfile.ZipFile(path) as z:
        tops = {p.split('/')[0] for p in z.namelist() if p.strip('/')}
        if tops != {'sprint53'}:
            raise SystemExit(f'!! {path} does not contain a single sprint53/ folder (found {tops}).')
        z.extractall('/content')
    print('   unpacked', path)

have_data = lambda: os.path.exists('/content/sprint53/results_taskB/B1_repeat_T07.jsonl')
have_code = lambda: os.path.exists('/content/sprint53/taskB/check_prompts_b5.py')
code_zips = []
while not (have_data() and have_code()):
    need = []
    if not have_data(): need.append('newest Stage 1 / run checkpoint (sprint53_taskB_run_<time>.zip)')
    if not have_code(): need.append('Step B5 code (sprint53_taskB_step5_code.zip)')
    print('Please upload ONE file now:', ' / '.join(need))
    for name in files.upload():
        unpack(name)
        if 'step5_code' in name.lower(): code_zips.append(name)
        else:
            for c in code_zips: unpack(c)
os.chdir('/content/sprint53')
print('\nReady. Task B result files:', len(glob.glob('results_taskB/*.jsonl')))


Please upload ONE file now: newest Stage 1 / run checkpoint (sprint53_taskB_run_<time>.zip) / Step B5 code (sprint53_taskB_step5_code.zip)


Saving sprint53_taskB_run_20261003_1852.zip to sprint53_taskB_run_20261003_1852.zip
   unpacked sprint53_taskB_run_20261003_1852.zip

Ready. Task B result files: 15


In [3]:
# 3. Install + start Ollama, download both judges, record versions, check the new prompts
%cd /content/sprint53
!bash colab/setup_ollama.sh
!python taskB/run_taskb.py --check
!python taskB/check_prompts_b5.py


/content
>> 1/4 System packages (zstd is needed by the Ollama installer, pciutils/lshw for GPU detection)
>> 2/4 Installing Ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>> 3/4 Starting the Ollama server in the background (log: /content/ollama.log)
{"version":"0.35.1"}
>> 4/4 Downloading judge models (same tags as Sprint 50)
success 
success 
NAME           ID              SIZE      MODIFIED               
qwen2.5:7b     845dbda0ea48    4.7 GB    Less than a second ago    
glm4:latest    5b699761eca5    5.5 GB    52 seconds ago            
>> Setup finished.
GPU            : Tesla T

In [4]:
# 4. Progress (first session: Stage 1b 2 phases, TOTAL 436, 0 done; Stage 2 needs Stage 1b first)
!python taskB/run_taskb.py --status --stage 1b
if os.path.exists('data/taskB/stage2_finalists.json'):
    !cat data/taskB/stage2_finalists.json
    !python taskB/run_taskb.py --status --stage 2


phase             calls   done     % malformed
B1b_C_T0            316    316  100%         0  ✔
B1b_C_repeat_T07    120    120  100%         0  ✔
TOTAL               436    436  100%
Next phase to run: none - stage complete
{
 "finalists": [
  "mqm_fewshot",
  "pairwise_tie"
 ],
 "C_passes_rule_vs_baseline": true,
 "C_not_clearly_worse_than_S2": false,
 "worse_than_S2": [
  "rho_human_setA (qwen) 0.81 -> 0.59",
  "planted_error_detected (glm4) 0.72 -> 0.58",
  "inconsistent_share (glm4) 0.63 -> 0.74"
 ],
 "note": "S2 and S5 approved by the user after Stage 1; C added only if it passes (pre-registered in Step B5)"
}phase             calls   done     % malformed
B2_T0               942    942  100%         0  ✔
B2_pairwise_T0      256    256  100%         0  ✔
B2_repeat_T07      1200    563   47%         0  …
B2_repeat_T0        480      0    0%         0   
B2_pairwise_T07     768      0    0%         0   
B2_wording          320      0    0%         0   
B2_wording_pairs    512      0

In [5]:
# 5. Checkpoint helper + run loop
import datetime, zipfile, os, time, json
from google.colab import files
def save_checkpoint():
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M')
    out = f'/content/sprint53_taskB_run_{stamp}.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, dirs, fnames in os.walk('/content/sprint53'):
            dirs[:] = [d for d in dirs if d not in ('node_modules', '.venv', '__pycache__')]
            for f in fnames:
                if f.endswith('.zip'): continue
                full = os.path.join(root, f); z.write(full, os.path.relpath(full, '/content'))
    print(f'Checkpoint: {out} ({os.path.getsize(out)/1e6:.1f} MB) - downloading...')
    files.download(out)

def run_stage(stage, session_budget_min=195, chunk_min=30):
    start = time.time()
    while True:
        left = session_budget_min - (time.time() - start) / 60
        if left < 3:
            print('Session budget used - keep the last checkpoint and continue next session.'); return False
        get_ipython().system(f'python taskB/run_taskb.py --run --stage {stage} --budget {int(min(chunk_min, left))}m')
        rc = get_ipython().user_ns.get('_exit_code', 0)
        save_checkpoint()
        if rc != 0:
            print('!! The runner stopped with an error (see above). Re-run cell 3, then this cell.'); return False
        if json.load(open(f'results_taskB/progress_stage{stage}.json'))['next_phase'] is None:
            print(f'STAGE {stage} COMPLETE'); return True
print('save_checkpoint() and run_stage() are ready')


save_checkpoint() and run_stage() are ready


In [ ]:
# 6. Part 1a - smoke test of the 10 new prompts (20 calls) + time estimates
!python taskB/run_taskb.py --smoke --prompts b5


>> Smoke test: 20 calls (10 prompts x 2 judges), models already loaded.
   [smoke] 20/20 calls  |  2.3 s/call  |  phase ETA 0m00s  |  session time left 59m13s

judge  prompt             parse         value tries   secs out tok    end
glm4   mqm_combined       ok               94     1   13.7      72   stop
glm4   mqm_rw2            ok               90     1    1.8      53   stop
glm4   mqm_fewshot_rw1    ok               94     1    2.6      72   stop
glm4   mqm_fewshot_rw2    ok               95     1    1.4      32   stop
glm4   mqm_combined_rw1   ok               95     1    1.9      44   stop
glm4   mqm_combined_rw2   ok               95     1    1.5      32   stop
glm4   pairwise_rw1       ok                A     1    0.5      12   stop
glm4   pairwise_rw2       ok              tie     1    0.5      12   stop
glm4   pairwise_tie_rw1   ok              tie     1    3.2      97   stop
glm4   pairwise_tie_rw2   ok              tie     1    3.5     104   stop
qwen   mqm_combined       

In [ ]:
# 7. Part 1b - Stage 1b: combined prompt C on the development half, then the analysis that decides
#    whether C joins Stage 2. Refuses to start if the smoke test had an unreadable or cut-off answer.
sm = [json.loads(l) for l in open('results_taskB/smoke_b5.jsonl')]
bad = [r['prompt_id'] + '/' + r['model'] for r in sm if r['parse_status'] == 'malformed' or r['done_reason'] == 'length']
if len(sm) != 20 or bad:
    print('!! Smoke test not clean:', bad or f'{len(sm)} calls', '- send the cell 6 output to Claude; do not continue.')
elif run_stage('1b'):
    !python taskB/analyze_stage1.py | sed -n '/Stage 1 decision rule/,$p'
    save_checkpoint()


>> Stage 1b: budget 30m00s, 1 worker(s)
== B1b_C_T0: 0/316 done, 316 to go
   [B1b_C_T0] 25/316 calls  |  3.2 s/call  |  phase ETA 15m30s  |  session time left 28m40s
   [B1b_C_T0] 50/316 calls  |  2.7 s/call  |  phase ETA 12m06s  |  session time left 27m43s
   [B1b_C_T0] 75/316 calls  |  2.8 s/call  |  phase ETA 11m14s  |  session time left 26m30s
   [B1b_C_T0] 100/316 calls  |  2.8 s/call  |  phase ETA 9m55s  |  session time left 25m24s
   [B1b_C_T0] 125/316 calls  |  2.9 s/call  |  phase ETA 9m12s  |  session time left 23m58s
   [B1b_C_T0] 150/316 calls  |  2.8 s/call  |  phase ETA 7m42s  |  session time left 23m01s
   [B1b_C_T0] 175/316 calls  |  2.7 s/call  |  phase ETA 6m22s  |  session time left 22m05s
   [B1b_C_T0] 200/316 calls  |  2.5 s/call  |  phase ETA 4m54s  |  session time left 21m32s
   [B1b_C_T0] 225/316 calls  |  2.4 s/call  |  phase ETA 3m39s  |  session time left 20m56s
   [B1b_C_T0] 250/316 calls  |  2.3 s/call  |  phase ETA 2m31s  |  session time left 20m26s
   [B

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

STAGE 1b COMPLETE
## Stage 1 decision rule

```
       strategy       prompt                                                                                                                       primary_better_for                                                                                                       clearly_worse  malformed stage1_rule
      S1 rubric  flat_rubric                                           inconsistent_share[glm4], inconsistent_share[qwen], mean_spread[qwen], mean_abs_judge_gap[ens] rho_true_band_setB (glm4) 0.54 -> 0.39; below_gold_strict (glm4) 0.78 -> 0.44; correct_below_85 (qwen) 0.06 -> 0.17      0.000        FAIL
     S4 acc+flu  flat_accflu                                                       below_gold_strict[glm4], below_gold_strict[qwen], literal_beats_fluent_wrong[glm4]                                                                              inconsistent_share (qwen) 0.53 -> 0.68      0.000        FAIL
    S2 few-shot  mqm_fewshot         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⚠️ **Stop here.** Run cell 8 only after results confirm the finalists.

In [6]:
# 8. Part 2 - Stage 2 on the held-out half (about 2.5-3.5 h; likely 2 sessions).
#    Next session: cells 1-5 (upload only the newest checkpoint), then this cell again - it continues.
!cat data/taskB/stage2_finalists.json
run_stage('2')


{
 "finalists": [
  "mqm_fewshot",
  "pairwise_tie"
 ],
 "C_passes_rule_vs_baseline": true,
 "C_not_clearly_worse_than_S2": false,
 "worse_than_S2": [
  "rho_human_setA (qwen) 0.81 -> 0.59",
  "planted_error_detected (glm4) 0.72 -> 0.58",
  "inconsistent_share (glm4) 0.63 -> 0.74"
 ],
 "note": "S2 and S5 approved by the user after Stage 1; C added only if it passes (pre-registered in Step B5)"
}>> Stage 2: budget 30m00s, 1 worker(s)
== B2_T0: already complete (942 calls)
== B2_pairwise_T0: already complete (256 calls)
== B2_repeat_T07: 563/1200 done, 637 to go
   [B2_repeat_T07] 25/637 calls  |  6.7 s/call  |  phase ETA 1h08m  |  session time left 27m12s
   [B2_repeat_T07] 50/637 calls  |  4.9 s/call  |  phase ETA 47m58s  |  session time left 25m54s
   [B2_repeat_T07] 75/637 calls  |  3.4 s/call  |  phase ETA 31m39s  |  session time left 25m46s
   [B2_repeat_T07] 100/637 calls  |  3.1 s/call  |  phase ETA 27m17s  |  session time left 24m54s
   [B2_repeat_T07] 125/637 calls  |  2.8 s/ca

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

>> Stage 2: budget 30m00s, 1 worker(s)
== B2_T0: already complete (942 calls)
== B2_pairwise_T0: already complete (256 calls)
== B2_repeat_T07: already complete (1200 calls)
== B2_repeat_T0: 215/480 done, 265 to go
   [B2_repeat_T0] 25/265 calls  |  3.1 s/call  |  phase ETA 12m31s  |  session time left 28m41s
   [B2_repeat_T0] 50/265 calls  |  1.7 s/call  |  phase ETA 6m11s  |  session time left 28m33s
   [B2_repeat_T0] 75/265 calls  |  1.6 s/call  |  phase ETA 4m57s  |  session time left 28m02s
   [B2_repeat_T0] 100/265 calls  |  1.7 s/call  |  phase ETA 4m40s  |  session time left 27m10s
   [B2_repeat_T0] 125/265 calls  |  1.8 s/call  |  phase ETA 4m07s  |  session time left 26m18s
   [B2_repeat_T0] 150/265 calls  |  1.7 s/call  |  phase ETA 3m10s  |  session time left 25m51s
   [B2_repeat_T0] 175/265 calls  |  1.5 s/call  |  phase ETA 2m12s  |  session time left 25m42s
   [B2_repeat_T0] 200/265 calls  |  1.5 s/call  |  phase ETA 1m36s  |  session time left 25m02s
   [B2_repeat_T0] 2

<IPython.core.display.Javascript object>

...

>> Stage 2: budget 30m00s, 1 worker(s)
== B2_T0: already complete (942 calls)
== B2_pairwise_T0: already complete (256 calls)
== B2_repeat_T07: already complete (1200 calls)
== B2_repeat_T0: already complete (480 calls)
== B2_pairwise_T07: already complete (768 calls)
== B2_wording: already complete (320 calls)
== B2_wording_pairs: 225/512 done, 287 to go
   [B2_wording_pairs] 25/287 calls  |  6.0 s/call  |  phase ETA 26m24s  |  session time left 27m28s
   [B2_wording_pairs] 50/287 calls  |  4.3 s/call  |  phase ETA 16m58s  |  session time left 26m25s
   [B2_wording_pairs] 75/287 calls  |  3.0 s/call  |  phase ETA 10m32s  |  session time left 26m16s
   [B2_wording_pairs] 100/287 calls  |  2.3 s/call  |  phase ETA 7m14s  |  session time left 26m07s
   [B2_wording_pairs] 125/287 calls  |  1.9 s/call  |  phase ETA 5m12s  |  session time left 25m58s
   [B2_wording_pairs] 150/287 calls  |  1.7 s/call  |  phase ETA 3m47s  |  session time left 25m50s
   [B2_wording_pairs] 175/287 calls  |  1.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

STAGE 2 COMPLETE


True

In [2]:
# 9. Manual checkpoint (run before closing the tab)
save_checkpoint()


NameError: name 'save_checkpoint' is not defined